In [1]:
!pip show tensorflow
!pip show pandas
!pip show numpy
!pip show scikit-learn
!pip show matplotlib
!pip show seaborn
!pip show Pillow
!pip show tqdm
!pip show transformers

Name: tensorflow
Version: 2.18.0
Summary: TensorFlow is an open source machine learning framework for everyone.
Home-page: https://www.tensorflow.org/
Author: Google Inc.
Author-email: packages@tensorflow.org
License: Apache 2.0
Location: D:\Python_mohinh\.venvrun\Lib\site-packages
Requires: tensorflow-intel
Required-by: 
Name: pandas
Version: 2.2.2
Summary: Powerful data structures for data analysis, time series, and statistics
Home-page: https://pandas.pydata.org
Author: 
Author-email: The Pandas Development Team <pandas-dev@python.org>
License: BSD 3-Clause License

Copyright (c) 2008-2011, AQR Capital Management, LLC, Lambda Foundry, Inc. and PyData Development Team
All rights reserved.

Copyright (c) 2011-2023, Open source contributors.

Redistribution and use in source and binary forms, with or without
modification, are permitted provided that the following conditions are met:

* Redistributions of source code must retain the above copyright notice, this
  list of conditions and 

In [4]:
pip show keras protobuf Flask librosa h5py

Name: keras
Version: 3.8.0
Summary: Multi-backend Keras
Home-page: 
Author: 
Author-email: Keras team <keras-users@googlegroups.com>
License: Apache License 2.0
Location: d:\Python_mohinh\.venvrun\Lib\site-packages
Requires: absl-py, h5py, ml-dtypes, namex, numpy, optree, packaging, rich
Required-by: tensorflow_intel
---
Name: protobuf
Version: 3.20.3
Summary: Protocol Buffers
Home-page: https://developers.google.com/protocol-buffers/
Author: 
Author-email: 
License: BSD-3-Clause
Location: d:\Python_mohinh\.venvrun\Lib\site-packages
Requires: 
Required-by: tensorboard, tensorflow_intel
---
Name: Flask
Version: 3.1.1
Summary: A simple framework for building complex web applications.
Home-page: 
Author: 
Author-email: 
License-Expression: BSD-3-Clause
Location: d:\Python_mohinh\.venvrun\Lib\site-packages
Requires: blinker, click, itsdangerous, jinja2, markupsafe, werkzeug
Required-by: 
---
Name: librosa
Version: 0.11.0
Summary: Python module for audio and music processing
Home-page: http

In [ ]:
# === 1. IMPORT CÁC THƯ VIỆN ===
print("Đang import các thư viện...")
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import Sequence, to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, roc_curve, auc, classification_report
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm
import warnings
import math
import gc
# Thư viện "chìa khóa" - Lần này sẽ import thành công
from transformers import TFAstModel

# === 2. THIẾT LẬP CÁC ĐƯỜNG DẪN (LOCAL CỦA BẠN) ===
METADATA_PATH = r"D:\Python_mohinh\data\mel_metadata_new_30s.csv"
IMAGE_DIR = r"D:\Python_mohinh\data\mel_spectrograms_db_30s"
SAVE_MODEL_PATH = r"D:\Python_mohinh\Mohinh\KG_Thu_Nghiem_12" # Đường dẫn xuất kết quả
os.makedirs(SAVE_MODEL_PATH, exist_ok=True)
print(f"Sẽ lưu kết quả vào: {SAVE_MODEL_PATH}")

# Tắt các cảnh báo
warnings.filterwarnings('ignore', category=UserWarning, module='tensorflow')
tf.get_logger().setLevel('ERROR')

# Cấu hình GPU (rất quan trọng khi chạy local)
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            # Bật 'memory growth' để TF không chiếm hết VRAM ngay lập tức
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Đang dùng {len(gpus)} GPU(s) với Memory Growth.")
    except RuntimeError as e:
        print(e)
else:
    print("Không tìm thấy GPU. Đang dùng CPU.")

# === 3. CÀI ĐẶT KÍCH THƯỚC CHO AST ===
TARGET_HEIGHT = 128 # 128 Mel-bins (chiều cao ảnh của bạn)
TARGET_WIDTH = 1024 # 1024 frame thời gian (AST yêu cầu)
TARGET_SHAPE = (TARGET_HEIGHT, TARGET_WIDTH, 3) 
BATCH_SIZE = 8 # Bắt đầu với 8 cho an toàn, bạn có thể tăng lên nếu VRAM cho phép
print(f"Dùng Batch Size: {BATCH_SIZE} (An toàn cho laptop cá nhân)")


# === 4. CALLBACK DỌN RÁC ===
class GarbageCollectorCallback(keras.callbacks.Callback):
    """Callback ép dọn RAM sau mỗi epoch"""
    def on_epoch_end(self, epoch, logs=None):
        print("\nThực thi dọn dẹp RAM (gc.collect())...")
        gc.collect()

# === 5. DATA GENERATOR CHO AST (ĐÃ SỬA LỖI) ===
class ASTDataGenerator(Sequence):
    """
    DataGenerator cho AST:
    1. Tải ảnh gốc 1292x128
    2. Cắt 1 lát 1024x128 (ngẫu nhiên khi train, ở giữa khi test)
    3. Chuẩn hóa & Stack 3 kênh
    """
    def __init__(self, metadata, image_dir, batch_size=BATCH_SIZE, 
                 target_height=TARGET_HEIGHT, target_width=TARGET_WIDTH, 
                 num_classes=8, augment=False, shuffle=True):
        self.metadata = metadata
        self.image_dir = image_dir
        self.batch_size = batch_size
        self.target_height = target_height
        self.target_width = target_width
        self.num_classes = num_classes
        self.augment = augment
        self.shuffle = shuffle
        self.on_epoch_end()

    def __len__(self):
        # Số batch mỗi epoch
        return int(np.floor(len(self.metadata) / self.batch_size))

    def __getitem__(self, index):
        # Lấy index cho batch
        indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        meta_batch = [self.metadata.iloc[k] for k in indexes]
        # Tạo dữ liệu
        X, y = self.__data_generation(meta_batch)
        return X, y

    def on_epoch_end(self):
        # Xáo trộn index cuối mỗi epoch
        self.indexes = np.arange(len(self.metadata))
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def get_all_labels(self):
        return np.array(self.metadata['genre'])

    def __data_generation(self, meta_batch):
        # Khởi tạo batch rỗng
        X = np.empty((self.batch_size, self.target_height, self.target_width, 3))
        y = np.empty((self.batch_size), dtype=int)

        # Lặp qua từng file trong batch
        for i, row in enumerate(meta_batch):
            filename = row['filename']
            label = row['genre']
            spec_path = os.path.join(self.image_dir, filename)
            
            try:
                # 1. Tải ảnh gốc
                img = Image.open(spec_path).convert('L') 
                img_np = np.array(img) # Shape (128, 1292)
                
                img_height, img_width = img_np.shape
                
                # 2. Logic Cắt (Crop) 1292 -> 1024
                if self.augment: # Train: Cắt ngẫu nhiên
                    max_start = img_width - self.target_width
                    start_x = np.random.randint(0, max(0, max_start) + 1)
                else: # Validation / Test: Cắt ở giữa
                    start_x = (img_width - self.target_width) // 2
                
                # Cắt lát (128, 1024)
                chunk = img_np[:, start_x : start_x + self.target_width] 
                
                # 3. Chuẩn hóa (Normalization)
                spec = np.log(chunk + 1e-6)
                mean = np.mean(spec)
                std = np.std(spec)
                spec = (spec - mean) / (std + 1e-6) # Sửa lỗi NaN
                
                # 4. Stack 3 kênh (AST yêu cầu)
                X[i,] = np.stack([spec, spec, spec], axis=-1) # (128, 1024, 3)
                y[i] = label
            except Exception as e:
                # Xử lý nếu file ảnh bị hỏng
                print(f"\nLỖI KHI TẢI FILE: {filename}. Lỗi: {e}. Dùng ảnh đen thay thế.")
                X[i,] = np.zeros((self.target_height, self.target_width, 3))
                y[i] = 0 # Gán tạm nhãn 0
        
        # 'return' nằm ngoài vòng lặp 'for' (đã sửa lỗi)
        return X, to_categorical(y, num_classes=self.num_classes)

# === 6. XÂY DỰNG MÔ HÌNH (AST) ===
def create_ast_model(num_classes=8, input_shape=TARGET_SHAPE):
    """
    Tạo mô hình Keras bằng cách dùng base model AST từ Hugging Face
    """
    # Tên model chuẩn, pre-train trên AudioSet (527 lớp)
    model_name = "MIT/ast-finetuned-audioset-10-10-0.4593"
    
    print(f"Đang tải base model AST ({model_name})...")
    # from_pt=True để tải trọng số PyTorch vào mô hình TF
    base_model = TFAstModel.from_pretrained(model_name, from_pt=True)
    print("Tải base model thành công.")
    
    # Giai đoạn 1: Đóng băng toàn bộ base model
    base_model.trainable = False
    
    # Xây dựng mô hình Keras
    inputs = layers.Input(shape=input_shape)
    # 'pooler_output' là output [Batch, 768] đã được chuẩn bị cho classification
    x = base_model(inputs).pooler_output 
    x = layers.Dropout(0.5)(x)
    # Thêm "đầu" 8 lớp của chúng ta
    outputs = layers.Dense(num_classes, activation='softmax')(x) 
    
    model = keras.Model(inputs, outputs)
    return model

# === 7. CÁC HÀM VẼ BIỂU ĐỒ ===

def plot_loss_accuracy(history1, history2, save_path):
    """Vẽ biểu đồ Loss/Acc của cả 2 giai đoạn"""
    
    # Gộp lịch sử từ 2 giai đoạn
    # Dùng .get() để tránh lỗi nếu một giai đoạn không chạy
    acc = history1.history.get('accuracy', []) + history2.history.get('accuracy', [])
    val_acc = history1.history.get('val_accuracy', []) + history2.history.get('val_accuracy', [])
    loss = history1.history.get('loss', []) + history2.history.get('loss', [])
    val_loss = history1.history.get('val_loss', []) + history2.history.get('val_loss', [])
    
    if not acc: # Kiểm tra nếu không có dữ liệu (ví dụ: bị dừng ngay)
        print("Không có dữ liệu lịch sử để vẽ biểu đồ Loss/Accuracy.")
        return

    stage1_epochs = len(history1.history.get('loss', []))
    
    plt.figure(figsize=(20, 8))
    
    # Biểu đồ Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(acc, label='Train Accuracy')
    plt.plot(val_acc, label='Validation Accuracy')
    if stage1_epochs > 0:
        plt.axvline(x=stage1_epochs - 1, color='red', linestyle='--', label='Bắt đầu Fine-Tuning')
    plt.title('AST Accuracy over Epochs (2 Giai đoạn)')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    
    # Biểu đồ Loss
    plt.subplot(1, 2, 2)
    plt.plot(loss, label='Train Loss')
    plt.plot(val_loss, label='Validation Loss')
    if stage1_epochs > 0:
        plt.axvline(x=stage1_epochs - 1, color='red', linestyle='--', label='Bắt đầu Fine-Tuning')
    plt.title('AST Loss over Epochs (2 Giai đoạn)')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    plt.savefig(os.path.join(save_path, "plot_ast_loss_accuracy_2_stages.png"))
    plt.show()

def plot_confusion_matrix(y_true, y_pred, classes, save_path, title="Confusion Matrix"):
    """Vẽ ma trận nhầm lẫn"""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
                xticklabels=classes, yticklabels=classes)
    plt.title(title)
    plt.ylabel("Nhãn Thật (True Label)")
    plt.xlabel("Nhãn Dự đoán (Predicted Label)")
    plt.savefig(os.path.join(save_path, "plot_ast_confusion_matrix.png"))
    plt.show()

def plot_roc_curve(y_true, y_score, classes, save_path):
    """Vẽ đường cong ROC (One-vs-Rest)"""
    n_classes = len(classes)
    y_true_bin = to_categorical(y_true, num_classes=n_classes)
    
    plt.figure(figsize=(10, 8))
    for i in range(n_classes):
        if len(np.unique(y_true_bin[:, i])) < 2:
            print(f"Bỏ qua ROC cho lớp {classes[i]} vì chỉ có 1 nhãn.")
            continue
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{classes[i]} (AUC = {roc_auc:.2f})")
        
    plt.plot([0, 1], [0, 1], "k--", label="Ngẫu nhiên (AUC = 0.50)")
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("Tỷ lệ Dương tính Giả (False Positive Rate)")
    plt.ylabel("Tỷ lệ Dương tính Thật (True Positive Rate)")
    plt.title("Đường cong ROC (AST - One-vs-Rest)")
    plt.legend(loc="lower right")
    plt.savefig(os.path.join(save_path, "plot_ast_roc_curve.png"))
    plt.show()

def hierarchical_error_analysis(y_true, y_pred, classes, save_path):
    """Vẽ biểu đồ Phân tích lỗi phân cấp"""
    # Đảm bảo 8 lớp này khớp với FMA-small
    groups = {
        "Acoustic/Natural": ["Folk", "Instrumental"],
        "Modern/Popular": ["Pop", "Hip-Hop"],
        "Rock/Alternative": ["Rock", "Experimental"],
        "Electronic/Global": ["Electronic", "International"]
    }
    group_map = {cls: grp for grp, cls_list in groups.items() for cls in cls_list}
    mapped_classes = group_map.keys()
    
    if not all(cls in mapped_classes for cls in classes):
        print(f"Cảnh báo: Tên lớp không khớp. Classes: {classes}. Bỏ qua Phân tích phân cấp.")
        return

    y_true_labels = [classes[y] for y in y_true]
    y_pred_labels = [classes[y] for y in y_pred]

    y_true_group = [group_map.get(label, 'Other') for label in y_true_labels]
    y_pred_group = [group_map.get(label, 'Other') for label in y_pred_labels]
    group_classes = list(groups.keys())
    
    cm_group = confusion_matrix(y_true_group, y_pred_group, labels=group_classes)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_group, annot=True, fmt="d", cmap="Greens", 
                xticklabels=group_classes, yticklabels=group_classes)
    plt.title("Ma trận nhầm lẫn Phân cấp (AST)")
    plt.ylabel("Nhóm Thật (True Group)")
    plt.xlabel("Nhóm Dự đoán (Predicted Group)")
    plt.savefig(os.path.join(save_path, "plot_ast_hierarchical_error.png"))
    plt.show()

# === 8. HÀM MAIN ĐỂ THỰC THI TOÀN BỘ QUY TRÌNH ===
def main():
    print("Đang tải metadata...")
    try:
        metadata = pd.read_csv(METADATA_PATH)
    except FileNotFoundError:
        print(f"LỖI: Không tìm thấy file metadata tại: {METADATA_PATH}")
        print("Vui lòng kiểm tra lại đường dẫn.")
        return

    label_encoder = LabelEncoder()
    metadata['genre'] = label_encoder.fit_transform(metadata['genre'])
    classes = label_encoder.classes_
    NUM_CLASSES = len(classes)
    print(f"Các lớp (thể loại): {classes}")

    # Split data
    train_meta, temp_meta = train_test_split(metadata, test_size=0.2, random_state=42, stratify=metadata['genre'])
    val_meta, test_meta = train_test_split(temp_meta, test_size=0.5, random_state=42, stratify=temp_meta['genre'])

    print(f"Tổng số mẫu: {len(metadata)}")
    print(f"Số mẫu Train: {len(train_meta)}")
    print(f"Số mẫu Validation: {len(val_meta)}")
    print(f"Số mẫu Test: {len(test_meta)}")

    # Chống thiên lệch
    class_weights_np = compute_class_weight(
        'balanced',
        classes=np.unique(metadata['genre']),
        y=metadata['genre']
    )
    class_weights = dict(enumerate(class_weights_np))
    print(f"Đã tính toán Class Weights: {class_weights}")

    # Tạo Generators
    print("Đang tạo Data Generators (AST)...")
    train_generator = ASTDataGenerator(
        train_meta, IMAGE_DIR, batch_size=BATCH_SIZE, num_classes=NUM_CLASSES, 
        augment=True, shuffle=True
    )
    val_generator = ASTDataGenerator(
        val_meta, IMAGE_DIR, batch_size=BATCH_SIZE, num_classes=NUM_CLASSES, 
        augment=False, shuffle=False
    )

    # === GIAI ĐOẠN 1: HUẤN LUYỆN "CÁI ĐẦU" (HEAD) ===
    print("\n" + "="*70)
    print("BẮT ĐẦU GIAI ĐOẠN 1: Huấn luyện 'cái đầu' phân loại (AST)")
    print("="*70)

    model = create_ast_model(num_classes=NUM_CLASSES)
    model.summary()

    # Callbacks
    model_checkpoint_1 = keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(SAVE_MODEL_PATH, "stage1_ast_best_head_model.keras"), 
        save_best_only=True,
        monitor='val_loss',
        verbose=1
    )
    early_stopping_1 = keras.callbacks.EarlyStopping(
        monitor='val_loss', 
        patience=5, # Dừng nếu val_loss không cải thiện sau 5 epoch
        restore_best_weights=True,
        verbose=1
    )
    reduce_lr_1 = keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', 
        patience=2, 
        factor=0.5,
        verbose=1
    )
    gc_callback = GarbageCollectorCallback()

    optimizer_1 = keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4)
    model.compile(
        optimizer=optimizer_1,
        loss='categorical_crossentropy', 
        metrics=['accuracy']
    )

    history_stage1 = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=50, # Sẽ dừng sớm
        callbacks=[early_stopping_1, reduce_lr_1, model_checkpoint_1, gc_callback],
        class_weight=class_weights, 
        verbose=1 
    )

    print("Giai đoạn 1 (AST Head) hoàn tất.")
    # Lưu lịch sử (phòng trường hợp crash)
    pd.DataFrame(history_stage1.history).to_csv(os.path.join(SAVE_MODEL_PATH, "history_stage1.csv"))

    # === GIAI ĐOẠN 2: TINH CHỈNH (FINE-TUNING) ===
    print("\n" + "="*70)
    print("BẮT ĐẦU GIAI ĐOẠN 2: Tinh chỉnh (Fine-Tuning) AST")
    print("="*70)

    # Mở băng base model
    base_model = model.layers[1] # Lớp 0 là Input, lớp 1 là TFAstModel
    base_model.trainable = True

    # AST có 12 lớp Transformer. Chúng ta mở băng 20% (khoảng 2-3 lớp)
    num_layers = len(base_model.transformer.layer)
    fine_tune_at = int(num_layers * 0.8) # Đóng băng 80% dưới

    # Đóng băng các lớp Embeddings
    base_model.embeddings.trainable = False
    # Đóng băng 80% lớp Transformer đầu tiên
    for layer in base_model.transformer.layer[:fine_tune_at]:
        layer.trainable = False

    print(f"AST có {num_layers} lớp Transformer.")
    print(f"Đã đóng băng {fine_tune_at} lớp dưới cùng.")
    print(f"Sẽ tinh chỉnh {num_layers - fine_tune_at} lớp trên cùng.")

    # Callbacks cho Giai đoạn 2
    model_checkpoint_2 = keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(SAVE_MODEL_PATH, "stage2_ast_best_finetuned_model.keras"),
        save_best_only=True,
        monitor='val_loss',
        verbose=1
    )
    early_stopping_2 = keras.callbacks.EarlyStopping(
        monitor='val_loss', 
        patience=5, # Dừng sớm 5 epoch
        restore_best_weights=True,
        verbose=1 
    )
    reduce_lr_2 = keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', 
        patience=2, 
        factor=0.5,
        verbose=1
    )

    # Compile Giai đoạn 2 (LR SIÊU NHỎ)
    optimizer_2 = keras.optimizers.AdamW(learning_rate=1e-5, weight_decay=1e-4) # Quan trọng
    model.compile(
        optimizer=optimizer_2,
        loss='categorical_crossentropy', 
        metrics=['accuracy']
    )

    model.summary()

    history_stage2 = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=50, # Sẽ dừng sớm
        callbacks=[early_stopping_2, reduce_lr_2, model_checkpoint_2, gc_callback],
        class_weight=class_weights, 
        verbose=1 
    )

    print("Giai đoạn 2 (Tinh chỉnh AST) hoàn tất.")
    pd.DataFrame(history_stage2.history).to_csv(os.path.join(SAVE_MODEL_PATH, "history_stage2.csv"))

    # === 9. ĐÁNH GIÁ CUỐI CÙNG ===
    print("\n" + "="*70)
    print("ĐÁNH GIÁ CUỐI CÙNG TRÊN TẬP TEST (MÔ HÌNH AST ĐÃ TINH CHỈNH)")
    print("="*70)

    print("Đang tải model AST tốt nhất đã được tinh chỉnh...")
    # Tải model tốt nhất từ Giai đoạn 2
    try:
        best_model = keras.models.load_model(
            os.path.join(SAVE_MODEL_PATH, "stage2_ast_best_finetuned_model.keras")
        )
    except Exception as e:
        print(f"LỖI: Không thể tải model. Lỗi: {e}")
        print("Có thể Giai đoạn 2 chưa chạy hoặc lưu model thất bại.")
        print("Đang thử tải model từ Giai đoạn 1...")
        try:
            best_model = keras.models.load_model(
                os.path.join(SAVE_MODEL_PATH, "stage1_ast_best_head_model.keras")
            )
            print("Đã tải model Giai đoạn 1. Kết quả sẽ chỉ dựa trên Giai đoạn 1.")
        except Exception as e2:
            print(f"LỖI: Cũng không thể tải model Giai đoạn 1. Lỗi: {e2}")
            return

    # Tạo generator riêng cho Test set (để đảm bảo không xáo trộn)
    print("Đang tạo Test Generator để đánh giá...")
    test_generator_eval = ASTDataGenerator(
        test_meta, IMAGE_DIR, batch_size=BATCH_SIZE, num_classes=NUM_CLASSES, 
        augment=False, shuffle=False # QUAN TRỌNG: shuffle=False
    )
    
    print("Dự đoán trên tập Test (AST)...")
    y_score = best_model.predict(test_generator_eval, verbose=1)

    y_pred = np.argmax(y_score, axis=1)

    # Lấy nhãn thật (y_true)
    print("Đang lấy nhãn thật từ Test Generator...")
    y_true = []
    for i in tqdm(range(len(test_generator_eval)), desc="Lấy nhãn thật"):
        _, y_batch = test_generator_eval[i]
        y_true.extend(np.argmax(y_batch, axis=1))

    y_true = np.array(y_true)
    y_true = y_true[:len(y_pred)] # Khớp độ dài (phòng trường hợp batch cuối)

    # --- BÁO CÁO PHÂN LOẠI ---
    print("\n" + "="*70)
    print("BÁO CÁO PHÂN LOẠI (AST) TRÊN TẬP TEST")
    print("="*70)
    print(classification_report(y_true, y_pred, target_names=classes))
    print("="*70)

    # --- VẼ TẤT CẢ BIỂU ĐỒ ---
    print("Đang tạo các biểu đồ trực quan hóa (AST)...")

    # 1. Biểu đồ Loss/Acc (2 Giai đoạn)
    try:
        hist1 = pd.read_csv(os.path.join(SAVE_MODEL_PATH, "history_stage1.csv")).to_dict('list')
        hist2 = pd.read_csv(os.path.join(SAVE_MODEL_PATH, "history_stage2.csv")).to_dict('list')
        plot_loss_accuracy(hist1, hist2, SAVE_MODEL_PATH)
    except FileNotFoundError:
        print("Không tìm thấy file history.csv. Bỏ qua vẽ biểu đồ Loss/Acc.")
        # Thử vẽ từ biến history (nếu script chạy liền mạch)
        if 'history_stage1' in locals() and 'history_stage2' in locals():
            print("Vẽ biểu đồ từ biến history trong bộ nhớ...")
            plot_loss_accuracy(history_stage1, history_stage2, SAVE_MODEL_PATH)

    # 2. Ma trận nhầm lẫn
    plot_confusion_matrix(y_true, y_pred, classes, SAVE_MODEL_PATH, 
                          title="Ma trận nhầm lẫn (AST - Sau Tinh chỉnh)")

    # 3. Đường cong ROC
    plot_roc_curve(y_true, y_score[:len(y_true)], classes, SAVE_MODEL_PATH) 

    # 4. Phân tích lỗi phân cấp
    hierarchical_error_analysis(y_true, y_pred, classes, SAVE_MODEL_PATH)

    print(f"\nHoàn tất! Kiểm tra thư mục {SAVE_MODEL_PATH} để xem kết quả.")

# === CHẠY TOÀN BỘ QUY TRÌNH ===
if __name__ == "__main__":
    main()

Đang import các thư viện...


ImportError: cannot import name 'TFAstModel' from 'transformers' (d:\Python_mohinh\.venv\Lib\site-packages\transformers\__init__.py)